# Extraction directinfo.ma — Renseignements juridiques par N° RC

Ce notebook (Google Colab) :
1. ouvre la **Recherche avancée** de www.directinfo.ma dans un navigateur Chromium sans interface (Playwright) ;
2. saisit le **Numéro RC** (et, en option, la ville) ;
3. **affiche le CAPTCHA** : vous tapez vous-même le code (pas de contournement automatique) ;
4. parcourt **toutes les pages de résultats**, ouvre chaque fiche `fiche-detail/...` ;
5. extrait le tableau **RENSEIGNEMENTS JURIDIQUES** (Dénomination, Statut juridique, Tribunal, N° de RC, Activité, Forme juridique, Date d'immatriculation, Capital…) ;
6. produit un fichier **Excel** téléchargeable.

> Respectez les conditions d'utilisation de directinfo.ma / OMPIC. Le script garde une pause entre chaque fiche.

**Mode d'emploi** : `Exécution > Tout exécuter`, puis répondez aux questions (N° RC, code CAPTCHA) dans la cellule 3.

In [ ]:
# 1) Installation (≈ 1 min, une seule fois par session Colab)
!pip -q install playwright beautifulsoup4 openpyxl pandas
!playwright install --with-deps chromium > /dev/null 2>&1 && echo "Chromium OK"

In [ ]:
# 2) Paramètres et fonctions
import asyncio, re, time, unicodedata
from urllib.parse import urljoin
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display, Image
from playwright.async_api import async_playwright

BASE_URL       = "https://www.directinfo.ma"
SEARCH_URL     = BASE_URL + "/recherche-multi-criteres"
PAUSE_SEC      = 1.5      # pause entre deux fiches (politesse envers le site)
MAX_PAGES      = 50       # garde-fou pagination
HEADLESS       = True

def norm(s: str) -> str:
    """minuscule, sans accents ni espaces superflus (pour comparer des libellés)."""
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().lower()

def clean(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

async def field_after_label(page, label_text, tag="input"):
    """Trouve le champ (input/select) qui suit un libellé visible, ex. 'Numéro RC'."""
    # a) <label for=...> correctement associé
    try:
        loc = page.get_by_label(label_text, exact=False)
        if await loc.count() > 0:
            return loc.first
    except Exception:
        pass
    # b) premier champ qui suit le texte du libellé dans le DOM
    xp = (f"xpath=//*[self::label or self::span or self::p or self::div]"
          f"[contains(normalize-space(.), '{label_text}') and not(.//{tag})]"
          f"/following::{tag}[not(@type='hidden')][1]")
    loc = page.locator(xp)
    if await loc.count() > 0:
        return loc.first
    raise RuntimeError(f"Champ introuvable pour le libellé « {label_text} »")

async def captcha_image(page):
    """Localise l'image du CAPTCHA."""
    for sel in ["img[src*='captcha' i]", "img[id*='captcha' i]", "img[class*='captcha' i]",
                "xpath=//*[contains(normalize-space(.),'Saisir le code')]/following::img[1]"]:
        loc = page.locator(sel)
        if await loc.count() > 0:
            return loc.first
    return None

def fiche_links(html):
    soup = BeautifulSoup(html, "html.parser")
    out = []
    for a in soup.select("a[href*='fiche-detail']"):
        url = urljoin(BASE_URL, a["href"])
        if url not in out:
            out.append(url)
    return out

def parse_renseignements(html, url):
    """Extrait le tableau 'RENSEIGNEMENTS JURIDIQUES' d'une fiche."""
    soup = BeautifulSoup(html, "html.parser")
    row = {"URL fiche": url}
    title = None
    for h in soup.find_all(["h1", "h2", "h3", "h4", "div", "span"]):
        if norm(h.get_text()) == "renseignements juridiques":
            title = h
            break
    table = title.find_next("table") if title else soup.find("table")
    if table is None:
        row["Erreur"] = "Tableau Renseignements juridiques introuvable"
        return row
    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        if len(cells) >= 2:
            key = clean(cells[0].get_text(" "))
            val = clean(cells[1].get_text(" "))
            if key:
                row[key] = val
    # Documents disponibles (Etats de synthèse, Statuts, …) affichés à droite du tableau
    docs = [clean(x.get_text(" ")) for x in soup.find_all(["a", "button"])
            if re.search(r"\(\d+\)\s*$", clean(x.get_text(" ")))]
    if docs:
        row["Documents disponibles"] = " | ".join(dict.fromkeys(docs))
    years = sorted(set(re.findall(r"Etats de synth[eè]se\s+(\d{4})", soup.get_text(" "))))
    if years:
        row["Années états de synthèse"] = ", ".join(years)
    return row

async def next_page(page):
    """Clique sur 'page suivante' si elle existe. Renvoie True si la page a changé."""
    candidates = ["a[rel='next']", "li.next:not(.disabled) a", ".pagination li.active + li a",
                  "a:has-text('Suivant')", "a:has-text('»')", "a:has-text('›')"]
    for sel in candidates:
        loc = page.locator(sel)
        if await loc.count() > 0 and await loc.first.is_visible():
            before = page.url + str(len(await page.content()))
            await loc.first.click()
            await page.wait_for_load_state("networkidle")
            after = page.url + str(len(await page.content()))
            return before != after
    return False

async def search_rc(rc, ville=None):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        ctx = await browser.new_context(locale="fr-FR", viewport={"width": 1400, "height": 1000})
        page = await ctx.new_page()

        # --- Formulaire de recherche (avec nouvelles tentatives si CAPTCHA faux) ---
        for attempt in range(1, 4):
            await page.goto(SEARCH_URL, wait_until="networkidle")
            await (await field_after_label(page, "Numéro RC")).fill(str(rc))
            if ville:
                v = await field_after_label(page, "Ville", tag="select")
                try:
                    await v.select_option(label=ville.upper())
                except Exception:
                    print(f"Ville « {ville} » non trouvée dans la liste : ignorée.")
            img = await captcha_image(page)
            if img is None:
                raise RuntimeError("Image CAPTCHA introuvable")
            display(Image(data=await img.screenshot()))
            code = input("Tapez le code affiché ci-dessus : ").strip()
            await (await field_after_label(page, "Saisir le code")).fill(code)
            await page.locator("button:has-text('Rechercher'), input[value='Rechercher']").first.click()
            await page.wait_for_load_state("networkidle")
            if "resultat" in page.url or fiche_links(await page.content()):
                break
            print(f"Recherche refusée (CAPTCHA incorrect ?) — tentative {attempt}/3")
        else:
            await browser.close()
            raise RuntimeError("Échec de la recherche après 3 tentatives")

        txt = (await page.content())
        m = re.search(r"Nombre de R[ée]sultats trouv[ée]s\s*:\s*(?:<[^>]+>)*\s*(\d+)", txt)
        print("Résultats annoncés par le site :", m.group(1) if m else "?")

        # --- Pagination : collecte de tous les liens de fiches ---
        links = []
        for n in range(MAX_PAGES):
            new = [u for u in fiche_links(await page.content()) if u not in links]
            links += new
            print(f"  page {n+1} : {len(new)} fiche(s)  (total {len(links)})")
            if not new or not await next_page(page):
                break

        # --- Visite de chaque fiche ---
        rows = []
        for i, url in enumerate(links, 1):
            try:
                await page.goto(url, wait_until="networkidle")
                r = parse_renseignements(await page.content(), url)
            except Exception as e:
                r = {"URL fiche": url, "Erreur": str(e)}
            rows.append(r)
            print(f"  [{i}/{len(links)}] {r.get('Dénomination', r.get('Erreur', ''))}")
            await asyncio.sleep(PAUSE_SEC)
        await browser.close()
        return rows

In [ ]:
# 3) Lancement : saisissez le N° RC (et éventuellement la ville / tribunal)
RC    = input("Numéro RC : ").strip()
VILLE = input("Ville (laisser vide pour toutes) : ").strip() or None
EXACT = input("Ne garder que les fiches dont le N° de RC est exactement égal ? (o/n) [o] : ").strip().lower() != "n"

rows = await search_rc(RC, VILLE)
df = pd.DataFrame(rows)
if EXACT and "N° de RC" in df.columns:
    df = df[df["N° de RC"].astype(str).str.strip() == RC]

# Ordre des colonnes : celles de la fiche en premier
ordre = ["Dénomination", "Statut juridique", "Tribunal", "N° de RC", "Activité", "Forme juridique",
         "Date d'immatriculation", "Capital (en MAD)", "Documents disponibles",
         "Années états de synthèse", "URL fiche", "Erreur"]
df = df[[c for c in ordre if c in df.columns] + [c for c in df.columns if c not in ordre]]
print(f"\n{len(df)} société(s) extraite(s)")
display(df)

In [ ]:
# 4) Export Excel + téléchargement
from google.colab import files
from openpyxl.utils import get_column_letter

nom = f"directinfo_RC_{RC}{'_' + VILLE if VILLE else ''}_{time.strftime('%Y%m%d_%H%M')}.xlsx"
with pd.ExcelWriter(nom, engine="openpyxl") as xw:
    df.to_excel(xw, index=False, sheet_name="Renseignements juridiques")
    ws = xw.sheets["Renseignements juridiques"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for i, col in enumerate(df.columns, 1):
        largeur = max([len(str(col))] + [len(str(v)) for v in df[col].fillna("")])
        ws.column_dimensions[get_column_letter(i)].width = min(largeur + 2, 60)
files.download(nom)

### En cas de problème
- **« Champ introuvable »** : le site a changé de mise en page. Ajoutez `await page.screenshot(path="debug.png")` dans `search_rc` pour voir la page.
- **Une seule page de résultats collectée alors que le site en annonce plus** : adaptez la liste `candidates` dans `next_page` au libellé du bouton de pagination réel.
- **Plusieurs RC à traiter** : relancez la cellule 3 pour chaque RC (un CAPTCHA par recherche), puis concaténez les DataFrames.